# 18. Speculative Decoding | 投机解码：打破推理的访存瓶颈 (Speculative Decoding)

**难度：** Medium | **标签：** `Inference`, `Memory Bound` | **目标人群：** 模型部署与推理引擎开发

在 16、17 节中，我们探讨了系统级优化（PagedAttention, SGLang）。本节我们将剖析推理领域近年来最大的**算法级**创新 —— **投机解码 (Speculative Decoding)**。

> **自回归生成的痛点：**
> 传统的大模型生成 Token 是一个一个蹦的。每生成一个 Token，庞大的模型参数就要从 GPU 的 HBM 读到 SRAM 里一次。这是非常的 **Memory Bound（访存受限）**。GPU 的算力几乎都在闲置等待数据搬运。

> **投机解码的破局：**
> 1. **草拟 (Draft)**：找一个非常小、速度极快的小模型（比如 1B 参数），让它连续盲猜并生成接下来 $K$ 个 Token（比如 4 个）。
> 2. **验证 (Verify)**：将这 $K$ 个草拟的 Token 一次性喂给庞大的目标模型（如 70B）。因为是并行计算，大模型验证这 4 个 Token 的时间，几乎和生成 1 个 Token 的时间一样短！
> 3. **接受与拒绝**：如果大模型的输出概率认同小模型的猜测，我们就直接免费获取了这 4 个 Token。如果不认同，我们在出错的地方截断，并用大模型的正确分布重采样。


### 动手实战：核心的接受/拒绝逻辑

面试中最常考的，就是如何对比草拟概率 $q(x)$ 和目标概率 $p(x)$ 来决定是否接受该 Token。
**算法法则**：
- 如果目标概率大于草拟概率 ($p \ge q$)，**100% 接受**。
- 如果目标概率小于草拟概率 ($p < q$)，以 $p/q$ 的概率接受它（丢硬币）。


In [3]:
import torch

In [8]:
def speculative_verify(draft_probs, target_probs, draft_tokens):
    """
    验证小模型生成的 K 个 Token，返回被接受的 Token 列表。
    
    Args:
        draft_probs: 小模型生成各个 token 时的概率预测分布, shape [K, vocab_size]
        target_probs: 大模型对这 K 个位置的真实概率预测分布, shape [K, vocab_size]
        draft_tokens: 小模型实际采样出的 K 个 token_id, shape [K]
        
    Returns:
        accepted_tokens: list, 最终被接受的 token_id 序列
    """
    K = len(draft_tokens)
    accepted_tokens = []
    
    for i in range(K):
        token_id = draft_tokens[i]
        
        # 提取目标概率 p 和草拟概率 q
        p = target_probs[i, token_id].item()
        q = draft_probs[i, token_id].item()
        
        # ==========================================
        # TODO 1: 判断是否 100% 接受
        # ==========================================
        if p >= q:
            accepted_tokens.append(token_id)
        # ==========================================
        # TODO 2: 以 p / q 的概率接受
        # 如果拒绝，立即中止整个验证循环！因为一步错步步错。
        # ==========================================
        else:
            r = torch.rand(1).item()
            if r < p / q:
                accepted_tokens.append(token_id)
            else:
                break
    
    return accepted_tokens



In [7]:
def test_speculative_decoding():
    try:
        torch.manual_seed(42)
        vocab_size = 100
        K = 4
        
        # 模拟生成
        draft_tokens = [10, 20, 30, 40]
        
        draft_probs = torch.rand(K, vocab_size)
        target_probs = torch.rand(K, vocab_size)
        
        # 强行设定：
        # 第 0 个 token: p > q (必接受)
        target_probs[0, 10] = 0.8
        draft_probs[0, 10] = 0.5
        
        # 第 1 个 token: p < q, 但随机数使得它刚好被接受 (p=0.4, q=0.5, p/q=0.8, rand设为0.5)
        target_probs[1, 20] = 0.4
        draft_probs[1, 20] = 0.5
        
        # 第 2 个 token: p 远小于 q，导致拒绝 (p=0.1, q=0.9, p/q=0.11, rand设为0.9)
        target_probs[2, 30] = 0.1
        draft_probs[2, 30] = 0.9
        
        original_rand = torch.rand
        def mock_rand(*args, **kwargs):
            # 依次返回 0.5, 0.9 供判断
            if not hasattr(mock_rand, 'call_count'):
                mock_rand.call_count = 0
            mock_rand.call_count += 1
            if mock_rand.call_count == 1:
                return torch.tensor([0.5])
            else:
                return torch.tensor([0.9])
        torch.rand = mock_rand
        
        accepted = speculative_verify(draft_probs, target_probs, draft_tokens)
        
        # 恢复
        torch.rand = original_rand
        
        assert accepted == [10, 20], f"期望只接受前两个 token，但得到 {accepted}"
        print("✅ 测试通过！投机解码逻辑实现通过测试。")
        
    except NotImplementedError:
        print("请先完成 TODO 代码。")
    except Exception as e:
        print(f"❌ 测试失败: {e}")

test_speculative_decoding()



✅ 测试通过！投机解码逻辑实现通过测试。


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码

In [ ]:
def speculative_verify(draft_probs, target_probs, draft_tokens):
    K = len(draft_tokens)
    accepted_tokens = []
    
    for i in range(K):
        token_id = draft_tokens[i]
        p = target_probs[i, token_id].item()
        q = draft_probs[i, token_id].item()
        
        if p >= q:
            accepted_tokens.append(token_id)
        else:
            r = torch.rand(1).item()
            if r < p / q:
                accepted_tokens.append(token_id)
            else:
                # 拒绝该 token，停止验证后续猜测
                break
                
    return accepted_tokens



### 解析

投机解码（Speculative Decoding）通过小模型草拟和大模型并行验证，将原本由于 Memory Bound 导致的计算等待时间转化为并发算力。验证时采用 $p/q$ 的接受概率，在数学上准确保证了即使经过了小模型的瞎猜，最终采样的 Token 分布依然和只用大模型自回归生成的分布严格一致，实现了“无损加速”。


## 进阶：拒绝后的残差重采样 (Residual Resampling) 与无损性证明

前面的 `speculative_verify` 在拒绝时直接 `break`，丢掉了"被拒绝的这一步"。但**完整的投机解码在拒绝处不会浪费**：它会用**残差分布 (residual distribution)** 重采样出一个修正 token 补上。这样保证**每一轮至少前进 1 个 token**，永不卡死。

### 残差分布的定义

当草拟 token $x$ 被拒绝时，从下面这个分布里重新采样：

$$p_{\text{res}}(x) = \frac{\max\big(0,\; p(x) - q(x)\big)}{\sum_{x'} \max\big(0,\; p(x') - q(x')\big)}$$

直觉：小模型 $q$ 在某些 token 上"高估"了（$q > p$），接受/拒绝步骤已经把这些多采的概率"退"掉了；残差分布只保留大模型 $p$ 比小模型 $q$ **多出来**的那部分概率质量（$p - q > 0$ 的地方），用它来补偿，正好把整体校正回 $p$。

### 无损性定理 (Lossless Guarantee)

> **定理**：设草拟分布为 $q$、目标分布为 $p$。采用如下采样流程：
> 1. 从 $q$ 采样候选 $x \sim q$；
> 2. 以概率 $\min\!\big(1, \frac{p(x)}{q(x)}\big)$ 接受 $x$；
> 3. 否则从残差分布 $p_{\text{res}}$ 重采样。
>
> 则最终输出 token $X$ 的分布**严格等于** $p$，即 $\Pr[X = x] = p(x)$ 对任意 $x$ 成立。

### 证明

最终得到某个具体 token $x$ 有两条互斥路径：**(A) 候选恰好是 $x$ 且被接受**，或 **(B) 候选被拒绝、重采样命中 $x$**。

**路径 A —— 接受 $x$：**
候选是 $x$ 的概率为 $q(x)$，接受概率为 $\min\!\big(1, \frac{p(x)}{q(x)}\big)$，于是

$$\Pr[\text{A}, X=x] = q(x)\cdot \min\!\Big(1, \tfrac{p(x)}{q(x)}\Big) = \min\big(q(x),\, p(x)\big).$$

**整体拒绝概率：**
对任意候选 $x'$，被拒绝的概率是 $q(x')\big(1 - \min(1, \tfrac{p(x')}{q(x')})\big) = q(x') - \min\big(q(x'), p(x')\big)$。求和：

$$\Pr[\text{reject}] = \sum_{x'}\Big(q(x') - \min(q(x'),p(x'))\Big) = 1 - \sum_{x'}\min\big(q(x'),p(x')\big),$$

这里用到了 $\sum_{x'} q(x') = 1$。

**残差分布的归一化常数：**

$$Z = \sum_{x'}\max\big(0, p(x')-q(x')\big) = \sum_{x'}\Big(p(x') - \min(p(x'),q(x'))\Big) = 1 - \sum_{x'}\min\big(p(x'),q(x')\big) = \Pr[\text{reject}].$$

（用到了恒等式 $\max(0, a-b) = a - \min(a,b)$ 以及 $\sum p = 1$。）**注意归一化常数恰好等于拒绝概率**——这正是残差重采样能无缝衔接的关键。

**路径 B —— 拒绝后重采样命中 $x$：**

$$\Pr[\text{B}, X=x] = \Pr[\text{reject}]\cdot p_{\text{res}}(x) = Z\cdot \frac{\max(0, p(x)-q(x))}{Z} = \max\big(0,\, p(x)-q(x)\big).$$

拒绝概率与归一化常数 $Z$ 直接约掉了。

**合并两条路径：**

$$\Pr[X=x] = \min\big(q(x),p(x)\big) + \max\big(0,\, p(x)-q(x)\big).$$

分两种情况：
- 若 $p(x) \ge q(x)$：$\;\min = q(x)$，$\;\max(0, p-q) = p(x)-q(x)$，相加 $= q(x) + p(x) - q(x) = p(x)$。✓
- 若 $p(x) < q(x)$：$\;\min = p(x)$，$\;\max(0, p-q) = 0$，相加 $= p(x)$。✓

两种情况都得到 $\Pr[X=x] = p(x)$。$\blacksquare$

**结论**：无论小模型 $q$ 怎么瞎猜，最终采样分布都和直接用大模型 $p$ 自回归采样**完全一致**——这就是"无损加速"的数学根基。小模型的质量只影响**接受率**（猜得越准、加速越多），但**不影响输出分布**。

### 为什么拒绝（重采样）之后必须 `break`，不能继续验证后面的 token？

因为**位置 $i$ 之后的所有草拟 token 及其验证概率，都是基于"位置 $i$ 用的是草拟 token $t_i$"这个前提算出来的。一旦 $t_i$ 被修正 token $t_i'$ 替换，后面的全部作废。** 这是自回归的本质：每个 token 的分布依赖它前面的**所有** token。

#### 把 $p$、$q$ 写成条件分布就清楚了

之前为了简洁写成 $p(x)$、$q(x)$，它们其实是**条件分布**：

$$q_i(x) = P_{\text{小}}(x \mid \text{prompt},\, t_1, \dots, t_{i-1}), \qquad p_i(x) = P_{\text{大}}(x \mid \text{prompt},\, t_1, \dots, t_{i-1})$$

大模型那一次并行前向，喂进去的是整条草拟序列 $[t_1, t_2, t_3, t_4]$，一次吐出每个位置的分布，但这些分布的**条件前缀**是固定的：

| 位置 | 大模型算出的 $p$ 条件在哪个前缀上 |
|------|-----------------------------|
| $p_1$ | prompt |
| $p_2$ | prompt, $t_1$ |
| $p_3$ | prompt, $t_1, t_2$ |
| $p_4$ | prompt, $t_1, t_2, t_3$ |

注意 $p_3$ 是**假设 $t_2$ 已经在上下文里**才算出来的。

#### 拒绝发生时会怎样

假设在位置 $i=2$ 拒绝了 $t_2$，残差重采样出修正 token $t_2' \ne t_2$。真实序列变成 $[t_1, t_2', \dots]$。此时：

- $p_3 = P_\text{大}(x \mid \text{prompt}, t_1, \mathbf{t_2})$ —— 条件在**旧的 $t_2$** 上；
- 但现在上下文已是 $\text{prompt}, t_1, \mathbf{t_2'}$ —— **前缀变了！**

$p_3$ 的"假设前提"已被推翻，它和 $q_3$ 一起彻底失效。位置 3、4 的所有概率都建立在一段**不再存在的历史**上。

> "一步错，步步错" —— 不是后面"可能"错，而是后面所有概率的**计算前提**都塌了。手里没有任何在新前缀 $t_2'$ 下的有效分布，自然无法继续验证。

#### 所以正确做法是

1. 本轮在拒绝处 `break`，输出 = 已接受的 $n$ 个 token + 1 个修正 token；
2. 用更新后的前缀 $[\text{prompt}, t_1, t_2']$，**下一轮**重新让小模型草拟、大模型并行验证。

#### 每轮的净收益

| 情况 | 本轮前进的 token 数 |
|------|------------------|
| 第 1 个就被拒 | 1（仅修正 token，仍有收益，不会卡死） |
| 接受 $n$ 个后被拒 | $n + 1$ |
| $K$ 个全部接受 | $K + 1$（最后 1 个来自大模型第 $K{+}1$ 位置的分布）|

所以**每轮稳赚 $\ge 1$ 个 token**，且只花了大模型 **1 次**前向。`break` 不是"放弃"，而是"这批草拟所依赖的前缀已被改写，必须拿新前缀重启一轮"。

In [12]:
def speculative_verify_full(draft_probs, target_probs, draft_tokens):
    """
    完整版投机解码：在拒绝处用残差分布重采样出一个修正 token。

    Args:
        draft_probs:  小模型预测分布, shape [K, vocab_size]
        target_probs: 大模型预测分布, shape [K, vocab_size]
        draft_tokens: 小模型采样出的 K 个 token_id, shape [K]

    Returns:
        output_tokens: list, 最终输出序列。
            - 接受了 n 个草拟 token；
            - 若发生拒绝，则在第 n 个之后追加 1 个残差重采样的修正 token；
            - 因此每一轮至少前进 1 个 token（n >= 0 时输出长度 >= 1，除非 K 个全部接受）。
    """
    K = len(draft_tokens)
    output_tokens = []

    for i in range(K):
        token_id = draft_tokens[i]
        p = target_probs[i, token_id].item()  # 目标概率 p(x)
        q = draft_probs[i, token_id].item()   # 草拟概率 q(x)

        if p >= q:
            # p >= q：100% 接受
            output_tokens.append(token_id)
        elif torch.rand(1).item() < p / q:
            # p < q：以 p/q 的概率接受
            output_tokens.append(token_id)
        else:
            # 拒绝：从残差分布 norm(max(0, p - q)) 重采样一个修正 token，然后中止
            residual = torch.clamp(target_probs[i] - draft_probs[i], min=0.0)
            residual = residual / residual.sum()        # 归一化，Z == 拒绝概率（见证明）
            corrected = torch.multinomial(residual, num_samples=1).item()
            output_tokens.append(corrected)
            break

    return output_tokens

In [ ]:
def test_lossless_distribution():
    """蒙特卡洛验证：投机采样(草拟 q + 接受/拒绝 + 残差重采样)的输出分布 == 目标分布 p。"""
    torch.manual_seed(0)
    vocab_size = 5

    # 随便造一对差异很大的分布 p (目标) 和 q (草拟)
    p = torch.softmax(torch.randn(vocab_size), dim=0)
    q = torch.softmax(torch.randn(vocab_size) * 2.0, dim=0)

    print("draft distribution", q)

    N = 200_000
    counts = torch.zeros(vocab_size)
    for _ in range(N):
        x = torch.multinomial(q, 1).item()          # 1) 从 q 采候选
        if torch.rand(1).item() < min(1.0, p[x].item() / q[x].item()):
            counts[x] += 1                           # 2) 以 min(1, p/q) 接受
        else:
            residual = torch.clamp(p - q, min=0.0)  # 3) 拒绝 -> 残差重采样
            residual = residual / residual.sum()
            counts[torch.multinomial(residual, 1).item()] += 1

    empirical = counts / N
    max_err = (empirical - p).abs().max().item()
    print("目标分布 p   :", [round(v, 3) for v in p.tolist()])
    print("投机采样分布 :", [round(v, 3) for v in empirical.tolist()])
    print(f"最大偏差     : {max_err:.4f}")
    assert max_err < 0.01, "分布偏差过大，无损性被破坏！"
    print("✅ 经验分布与目标分布 p 一致，无损性得证（蒙特卡洛验证）。")

test_lossless_distribution()